# HIG UI-Element Detector — YOLO training

Trains a YOLO11 object detector on the VINS iPhone dataset (10 UI classes) at **1280 px**, so small text and icons stay visible, then exports a **Core ML** model for the iOS app.

**Before you start**
1. Upload `VINS_YOLO.zip` (from the OddessyCH1 folder) to Google Drive, into a folder called **`HIG`** → `My Drive/HIG/VINS_YOLO.zip`.
2. In Colab: **Runtime → Change runtime type → T4 GPU** (or better).
3. Run the cells top to bottom. Training results are saved to Drive, so a disconnect doesn't lose progress — use the **Resume** cell.

In [4]:
# 1. Check the GPU
!nvidia-smi

Thu Oct  8 14:09:13 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   44C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [5]:
# 2. Settings — change these if you like
DRIVE_DIR  = "/content/drive/MyDrive/HIG"   # where VINS_YOLO.zip lives and results are saved
MODEL      = "yolo11s.pt"   # n = fastest, s = good balance, m = more accurate but heavier on iPhone
IMGSZ      = 1280           # training resolution (keeps text ~20 px tall)
EPOCHS     = 100
RUN_NAME   = "vins_yolo11s_1280"

In [6]:
# 3. Mount Google Drive and unpack the dataset
from google.colab import drive
drive.mount("/content/drive")

import os, zipfile, pathlib
zip_path = f"{DRIVE_DIR}/VINS_YOLO.zip"
assert os.path.exists(zip_path), f"Upload VINS_YOLO.zip to {DRIVE_DIR} first"
if not os.path.exists("/content/VINS_YOLO"):
    zipfile.ZipFile(zip_path).extractall("/content")

# Point data.yaml at the absolute path so Ultralytics finds it
DATA = "/content/VINS_YOLO/data.yaml"
y = pathlib.Path(DATA).read_text().splitlines()
y = ["path: /content/VINS_YOLO" if l.startswith("path:") else l for l in y]
pathlib.Path(DATA).write_text("\n".join(y) + "\n")
print(pathlib.Path(DATA).read_text())
for s in ["train", "val", "test"]:
    print(s, len(os.listdir(f"/content/VINS_YOLO/images/{s}")), "images")

Mounted at /content/drive
path: /content/VINS_YOLO
train: images/train
val: images/val
test: images/test
names:
  0: BackgroundImage
  1: CheckedTextView
  2: EditText
  3: Icon
  4: Image
  5: PageIndicator
  6: Switch
  7: Text
  8: TextButton
  9: UpperTaskBar

train 965 images
val 120 images
test 122 images


In [7]:
# 4. Install Ultralytics
!pip install -q ultralytics
import ultralytics; ultralytics.checks()

Ultralytics 8.4.174 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
Setup complete ✅ (2 CPUs, 12.7 GB RAM, 43.7/112.6 GB disk)


## Train
- `fliplr=0` — no mirror augmentation: UI layouts aren't symmetric (back buttons sit on the left).
- `batch=-1` picks the largest batch that fits the GPU.
- `patience=20` stops early if validation accuracy stops improving.
- Results go to `My Drive/HIG/runs/<RUN_NAME>`.

In [8]:
# 5. Train
from ultralytics import YOLO
model = YOLO(MODEL)
model.train(
    data=DATA, imgsz=IMGSZ, epochs=EPOCHS, patience=20, batch=-1,
    fliplr=0.0,
    project=f"{DRIVE_DIR}/runs", name=RUN_NAME, exist_ok=True,
    plots=True,
)

Ultralytics 8.4.174 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/VINS_YOLO/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1280, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=vins_yolo11s_1280, nbs=64, nms=None, opset=None, optimize=Fa

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7ea8398d8d70>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.0

In [9]:
# 5b. ONLY if Colab disconnected mid-training: resume from the last checkpoint
from ultralytics import YOLO
model = YOLO(f"{DRIVE_DIR}/runs/{RUN_NAME}/weights/last.pt")
model.train(resume=True)

WARNING ⚠️ model '/content/drive/MyDrive/HIG/runs/vins_yolo11s_1280/weights/last.pt' is not a resumable training checkpoint (missing epoch/optimizer state). Use 'resume' only to continue incomplete training. Starting new training instead.
Ultralytics 8.4.174 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=coco8.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1280, iou=0.7, kob

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([ 0, 16, 17, 20, 25, 58])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7ea8e4453b60>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,

In [10]:
# 6. Evaluate the best model on the held-out TEST set (per-class accuracy)
from ultralytics import YOLO
best = YOLO(f"{DRIVE_DIR}/runs/{RUN_NAME}/weights/best.pt")
m = best.val(data=DATA, split="test", imgsz=IMGSZ, plots=True,
             project=f"{DRIVE_DIR}/runs", name=f"{RUN_NAME}_test", exist_ok=True)
print(f"\nmAP50: {m.box.map50:.3f}   mAP50-95: {m.box.map:.3f}\n")
print(f"{'class':<16}{'AP50':>8}{'AP50-95':>9}")
for j, ci in enumerate(m.box.ap_class_index):      # only classes present in the test set
    print(f"{best.names[int(ci)]:<16}{m.box.ap50[j]:>8.3f}{m.box.ap[j]:>9.3f}")

Ultralytics 8.4.174 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
YOLO11s summary (fused): 100 layers, 9,416,670 parameters, 0 gradients, 21.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 96.9±51.2 MB/s, size: 448.5 KB)
val: Scanning /content/VINS_YOLO/labels/test... 122 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 122/122 403.2it/s 0.3s
val: New cache created: /content/VINS_YOLO/labels/test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 1.2s/it 9.8s
                   all        122       2386      0.868       0.91      0.929      0.815
       BackgroundImage          6          6       0.56        0.5      0.642      0.267
       CheckedTextView          4          4      0.457       0.75      0.745      0.693
              EditText         41         71      0.925      0.986      0.986      0.946
                  Icon         95        532      0.938      0.989      0.983    

In [11]:
# 7. Look at predictions on a few test screenshots
import glob
from IPython.display import Image as IPImage, display
imgs = sorted(glob.glob("/content/VINS_YOLO/images/test/*.jpg"))[:6]
res = best.predict(imgs, imgsz=IMGSZ, conf=0.25, save=True,
                   project=f"{DRIVE_DIR}/runs", name=f"{RUN_NAME}_preview", exist_ok=True)
for p in sorted(glob.glob(f"{DRIVE_DIR}/runs/{RUN_NAME}_preview/*.jpg"))[:6]:
    display(IPImage(filename=p, width=320))

Output hidden; open in https://colab.research.google.com to view.

## Export to Core ML
Exported at **1280 × 736** (portrait, close to an iPhone screen's shape) with NMS built in, so in Swift it works like the Create ML model: Vision returns `VNRecognizedObjectObservation`s.
In Swift set `request.imageCropAndScaleOption = .scaleFill`.

In [12]:
# 8. Export Core ML model and copy it to Drive
import shutil
# half=False is required: 16-bit compression fails on YOLO11 attention layers
path = best.export(format="coreml", imgsz=[1280, 736], nms=True, half=False)
dst = f"{DRIVE_DIR}/HIGElementDetector.mlpackage"
shutil.rmtree(dst, ignore_errors=True)
shutil.copytree(path, dst)
print("Saved", dst, "— download it and drag it into Xcode")

WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
Ultralytics 8.4.174 🚀 Python-3.13.15 torch-2.11.0+cu130 CPU (Intel Xeon CPU @ 2.00GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino
YOLO11s summary (fused): 100 layers, 9,416,670 parameters, 0 gradients, 49.7 GFLOPs

PyTorch: starting from '/content/drive/MyDrive/HIG/runs/vins_yolo11s_1280/weights/best.pt' with input shape (1, 3, 1280, 736) BCHW and output shape(s) (1, 14, 19320) (18.4 MB)
requirements: Ultralytics requirement ['coremltools>=9.0'] not found, attempting AutoUpdate...
Using Python 3.13.15 environment at: /usr
Resolved 12 packages in 364ms
Prepared 3 packages in 129ms
Installed 3 packages in 13ms
 + cattrs==26.2.1
 + coremltools==9.0
 + pyaml==26.7.0

requirements: AutoUpdate success ✅ 0.9s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect



/usr/local/lib/python3.13/dist-packages/coremltools/optimize/torch/palettization/fake_palettize.py:82: SyntaxWarning: invalid escape sequence '\_'
  n_bits (:obj:`int`): Number of palettization bits. There would be :math:`2^{n\_bits}` unique weights in the ``LUT``.



CoreML: starting export with coremltools 9.0...


Running MIL default pipeline:  11%|█         | 10/95 [00:00<00:01, 43.30 passes/s]/usr/local/lib/python3.13/dist-packages/coremltools/converters/mil/mil/passes/defs/preprocess.py:273: UserWarning: Output, '1150', of the source model, has been renamed to 'var_1150' in the Core ML model.
  warnings.warn(msg.format(var.name, new_name))
/usr/local/lib/python3.13/dist-packages/coremltools/converters/mil/mil/passes/defs/preprocess.py:273: UserWarning: Output, '1152', of the source model, has been renamed to 'var_1152' in the Core ML model.
  warnings.warn(msg.format(var.name, new_name))
Running MIL backend_mlprogram pipeline: 100%|██████████| 12/12 [00:00<00:00, 80.99 passes/s]


CoreML: starting pipeline with coremltools 9.0...
CoreML: pipeline success
CoreML: export success ✅ 24.1s, saved as '/content/drive/MyDrive/HIG/runs/vins_yolo11s_1280/weights/best.mlpackage' (18.2 MB)

Export complete (25.3s)
Results saved to /content/drive/MyDrive/HIG/runs/vins_yolo11s_1280/weights/best.mlpackage
Predict:         yolo predict task=detect model=/content/drive/MyDrive/HIG/runs/vins_yolo11s_1280/weights/best.mlpackage imgsz=1280,736 
Validate:        yolo val task=detect model=/content/drive/MyDrive/HIG/runs/vins_yolo11s_1280/weights/best.mlpackage imgsz=1280,736 data=/content/VINS_YOLO/data.yaml  WARNING ⚠️ non-PyTorch val requires square images, 'imgsz=[1280, 736]' will not work. Use export 'imgsz=1280' if val is required.
Visualize:       https://netron.app
Saved /content/drive/MyDrive/HIG/HIGElementDetector.mlpackage — download it and drag it into Xcode
